# Orbit calculation post processing and plotting

In [ ]:
%matplotlib ipympl
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import altair as alt
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import plotly.io as pio
pio.renderers.default = "notebook"  # or "browser"

## Data loading and preprocessing

In [ ]:
subharmonics_colors = {
    0: "#BAC24C",
    1: "#5179D6",
    2: "#2ca02c",
    3: "#d62728",
    4: "#9467bd",
    5: "#9651D6",
}


In [ ]:
orbits = pl.read_parquet("outputs/orbits.parquet")
orbits

In [ ]:
orbit_data = pl.read_parquet("outputs/orbit_data.parquet")
orbit_data.sort("Ph", descending=True)

In [ ]:

chart = (
    alt.Chart(orbit_data.with_columns((pl.col("Ph") * 1000.0).alias("Ph_scaled")))
    .mark_circle(size=100)
    .encode(
        x=alt.X("fd:Q").title("Drive Frequency [Hz]"),
        y=alt.Y("Ph_scaled").scale(type="log").title("Harvested Power [mW]"),
        # y=alt.Y("Ph_scaled").title("Harvested Power [mW]"),
        color=alt.Color(
            "detected_subharmonic:N",
            title="Detected subharmonic",
            scale=alt.Scale(
                domain=list(subharmonics_colors.keys()),
                range=list(subharmonics_colors.values()),
            ),
        ),
        shape="detected_subharmonic:N",
        tooltip=["orbit_label", "fd", "Ph", "detected_subharmonic"],
    )
    .properties(title="Damping dissipation vs. Drive Frequency", width=1000, height=700)
).interactive()

chart.show()

## Orbits a given frequency

In [ ]:


fd = 50.0
x = "x"
y = "dotx"
z = "v"
xlabel = "Position, x"
ylabel = "Speed, dot x"
zlabel = "Voltage, v"

orbits_fd = orbits.filter(pl.col("fd") == fd)
orbits_data_fd = orbit_data.filter(pl.col("fd") == fd)
orbits_dic = {}
for row in orbits_fd.iter_rows(named=True):
    olabel = row["orbit_label"]
    alabel = row["attractor_label"]
    if olabel not in orbits_dic.keys():
        orbits_dic[olabel] = {"orbit": {}, "data": row}
    odata = pl.read_parquet(
        f"outputs/orbits_from_attractors/orbit_{olabel}_attractor_{alabel}.parquet"
    )
    orbits_dic[olabel]["orbit"][alabel] = odata


fig = go.Figure()

for ok, odata in orbits_dic.items():
    xv = []
    yv = []
    zv = []
    name = ""
    sh = odata["data"]["detected_subharmonic"]
    if sh == 1:
        name += "H"
    else:
        name += f"SH"
    name += f"{sh}_id{ok}"
    for ak, adata in odata["orbit"].items():
        orbit = odata["orbit"][ak].to_pandas()    
        xv.append(orbit[x].values)
        yv.append(orbit[y].values)
        zv.append(orbit[z].values)
    xv = np.concatenate(xv)
    yv = np.concatenate(yv)
    zv = np.concatenate(zv)
    Np = len(xv) // sh
    fig.add_trace(go.Scatter3d(
            x=xv,
            y=yv,
            z=zv,
            mode="lines",
            legendgroup=f"g{ok}",
            line=dict(
                color=subharmonics_colors[odata["data"]["detected_subharmonic"]], width=4
            ),
            name=name,
        )
    )
    fig.add_trace(go.Scatter3d(
            x=xv[::Np],
            y=yv[::Np],
            z=zv[::Np],
            mode="markers",
            legendgroup=f"g{ok}",
            marker=dict(symbol="circle", color=subharmonics_colors[odata["data"]["detected_subharmonic"]], size=3),
            showlegend=False
        )
    )

fig.update_layout(
    title=f"Orbits and attractors at Drive Frequency fd={fd} Hz",
    legend=dict(
        title="Orbits and attractors",
        x=0.02, y=0.98,
        bgcolor="rgba(255,255,255,0.7)"
    ),
    margin=dict(l=0, r=0, t=50, b=0),

    scene=dict(
        xaxis=dict(
            title=xlabel,
            showgrid=True,
            gridcolor="rgba(0,0,0,0.15)",
            zeroline=True,
            zerolinecolor="rgba(0,0,0,0.25)",
            showbackground=True,
            backgroundcolor="rgba(245,245,245,1)",
            ticks="outside",
        ),
        yaxis=dict(
            title=ylabel,
            showgrid=True,
            gridcolor="rgba(0,0,0,0.15)",
            zeroline=True,
            showbackground=True,
            backgroundcolor="rgba(245,245,245,1)",
        ),
        zaxis=dict(
            title=zlabel,
            showgrid=True,
            gridcolor="rgba(0,0,0,0.15)",
            zeroline=True,
            showbackground=True,
            backgroundcolor="rgba(245,245,245,1)",
        ),

        # Keep scales comparable (optional)
        aspectmode="cube",     # or "data" / "manual"
        # aspectratio=dict(x=1, y=1, z=0.6),

        # Initial view (optional)
        camera=dict(
            eye=dict(x=1.6, y=1.6, z=1.1)
        )
    )
)

fig.update_layout(
    width=1000,
    height=1000,
)
fig.show()

## Orbits as a function of frequency

In [ ]:
x = "x"
y = "dotx"
xlabel = "Position, x"
ylabel = "Speed, dot x"
zlabel = "Drive Frequency, fd [Hz]"


orbits_dic = {}
for row in orbits.iter_rows(named=True):
    olabel = row["orbit_label"]
    alabel = row["attractor_label"]
    if olabel not in orbits_dic.keys():
        orbits_dic[olabel] = {"orbit": {}, "data": row}
    odata = pl.read_parquet(
        f"outputs/orbits_from_attractors/orbit_{olabel}_attractor_{alabel}.parquet"
    )
    orbits_dic[olabel]["orbit"][alabel] = odata


fig = go.Figure()

show_legend = set()
for ok, odata in orbits_dic.items():
    xv = []
    yv = []
   
    name = ""
    sh = odata["data"]["detected_subharmonic"]
    fd = odata["data"]["fd"]
    if sh == 1:
        name += "H"
    else:
        name += f"SH"
    name += f"{sh}"
    for ak, adata in odata["orbit"].items():
        orbit = odata["orbit"][ak].to_pandas()    
        xv.append(orbit[x].values)
        yv.append(orbit[y].values)
    xv = np.concatenate(xv)
    yv = np.concatenate(yv)
    Np = len(xv) // sh
    zv = np.ones_like(xv) * fd
    oname = name + "_orbit"
    aname = name + "_attractor"
    fig.add_trace(go.Scatter3d(
            x=xv,
            y=yv,
            z=zv,
            mode="lines",
            legendgroup=oname,
            line=dict(
                color=subharmonics_colors[odata["data"]["detected_subharmonic"]], width=4
            ),
            name=oname,
            showlegend=oname not in show_legend,
        )
    )   
    show_legend.add(oname)
    fig.add_trace(go.Scatter3d(
            x=xv[::Np],
            y=yv[::Np],
            z=zv[::Np],
            mode="markers",
            legendgroup=aname,
            name =aname,
            marker=dict(symbol="circle", color=subharmonics_colors[odata["data"]["detected_subharmonic"]], size=3),
            showlegend=aname not in show_legend
        )
    )
    show_legend.add(aname)

fig.update_layout(
    title=f"Orbits and attractors at Drive Frequency fd={fd} Hz",
    legend=dict(
        title="Orbits and attractors",
        x=0.02, y=0.98,
        bgcolor="rgba(255,255,255,0.7)"
    ),
    margin=dict(l=0, r=0, t=50, b=0),

    scene=dict(
        xaxis=dict(
            title=xlabel,
            showgrid=True,
            gridcolor="rgba(0,0,0,0.15)",
            zeroline=True,
            zerolinecolor="rgba(0,0,0,0.25)",
            showbackground=True,
            backgroundcolor="rgba(245,245,245,1)",
            ticks="outside",
        ),
        yaxis=dict(
            title=ylabel,
            showgrid=True,
            gridcolor="rgba(0,0,0,0.15)",
            zeroline=True,
            showbackground=True,
            backgroundcolor="rgba(245,245,245,1)",
        ),
        zaxis=dict(
            title=zlabel,
            showgrid=True,
            gridcolor="rgba(0,0,0,0.15)",
            zeroline=True,
            showbackground=True,
            backgroundcolor="rgba(245,245,245,1)",
        ),

        # Keep scales comparable (optional)
        aspectmode="cube",     # or "data" / "manual"
        # aspectratio=dict(x=1, y=1, z=0.6),

        # Initial view (optional)
        camera=dict(
            eye=dict(x=1.6, y=1.6, z=1.1)
        )
    )
)

fig.update_layout(
    width=1000,
    height=700,
)
fig.show()